# **Voice Changer**

# **This is an fork of (@razzytest)'s code, from this forked code you can run "Public-W-okada-Voice-Changer" A Realtime Voice Changer without any issues, you just have to run pyworld separately.**

# :D

# **Enjoy Your Voice Changer!**

In [ ]:
!pip install pyworld

In [ ]:
# This will make that we're on the right folder before installing
%cd /kaggle/working/


!pip install colorama --quiet
from colorama import Fore, Style
import os

print(f"{Fore.CYAN}> Cloning the repository...{Style.RESET_ALL}")
!git clone  https://github.com/BadKiko/voice-changer-collab.git  --quiet
print(f"{Fore.GREEN}> Successfully cloned the repository!{Style.RESET_ALL}")

%cd voice-changer-collab/server/

print(f"{Fore.CYAN}> Installing libportaudio2...{Style.RESET_ALL}")
!apt-get -y install libportaudio2 -qq

print(f"{Fore.CYAN}> Installing pre-dependencies...{Style.RESET_ALL}")
# Install dependencies that are missing from requirements.txt and pyngrok
!pip install faiss-gpu fairseq pyngrok --quiet
print(f"{Fore.CYAN}> Installing dependencies from requirements.txt...{Style.RESET_ALL}")
!pip install -r requirements.txt --quiet
# Downgrade torch to version 2.0.0 to match torchdata 0.6.0 requirements
!pip install torch==2.0.0

# Install torchdata 0.6.0
!pip install torchdata==0.6.0

# Verify the installation
import torch
import torchdata
print(f"Torch version: {torch.__version__}")
print(f"Torchdata version: {torchdata.__version__}")

print(f"{Fore.GREEN}> Successfully installed all packages!{Style.RESET_ALL}")
print(f"{Fore.GREEN}> You can safely ignore the dependency conflict errors, it's a error from Kaggle and don't interfer on Voice Changer!{Style.RESET_ALL}")

In [ ]:
# @title Start Server **using ngrok**
# @markdown

# @markdown
Token = '2oycoct0WKtqhqaCsbtt6Pq6hBi_2S2bjJQBqxCRHF9BrswsP' # @param {type:"string"}
# @markdown
Region = "in - India (Mumbai)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)","eu - Europe (Frankfurt)", "in - India (Mumbai)","jp - Japan (Tokyo)","sa - South America (Sao Paulo)", "us - United States (Ohio)"]

# @markdown
ClearConsole = True  # @param {type:"boolean"}

# ---------------------------------
# DO NOT TOUCH ANYTHING DOWN BELOW!
# ---------------------------------

%cd /kaggle/working/voice-changer-collab/server

from pyngrok import conf, ngrok
MyConfig = conf.PyngrokConfig()
MyConfig.auth_token = Token
MyConfig.region = Region[0:2]
#conf.get_default().authtoken = Token
#conf.get_default().region = Region
conf.set_default(MyConfig);

import subprocess, threading, time, socket, urllib.request
PORT = 8000

from pyngrok import ngrok
ngrokConnection = ngrok.connect(PORT)
public_url = ngrokConnection.public_url

from IPython.display import clear_output

def wait_for_server():
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        if result == 0:
            break
        sock.close()
    if ClearConsole:
        clear_output()
    print(public_url)
threading.Thread(target=wait_for_server, daemon=True).start()

!python3 MServ.py \
  -p {PORT} \
  --https False \
  --content_vec_500 pretrain/checkpoint_best_legacy_500.pt \
  --content_vec_500_onnx pretrain/content_vec_500.onnx \
  --content_vec_500_onnx_on true \
  --hubert_base pretrain/hubert_base.pt \
  --hubert_base_jp pretrain/rinna_hubert_base_jp.pt \
  --hubert_soft pretrain/hubert/hubert-soft-0d54a1f4.pt \
  --nsf_hifigan pretrain/nsf_hifigan/model \
  --crepe_onnx_full pretrain/crepe_onnx_full.onnx \
  --crepe_onnx_tiny pretrain/crepe_onnx_tiny.onnx \
  --rmvpe pretrain/rmvpe.pt \
  --model_dir model_dir \
  --samples samples.json

ngrok.disconnect(ngrokConnection.public_url)